<a href="https://colab.research.google.com/github/tsakailab/MultivariateAnalysis/blob/main/ipynb/ex_LinearRegressionPrimer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 例題で理解する回帰分析

* データが与えられたとき，モデル変数 $\tilde{\boldsymbol{\beta}}$ の線形回帰モデルによる予測の誤差（残差）$\boldsymbol{\delta}$ は，
データで決まる $\tilde{\boldsymbol{X}}$ と $\boldsymbol{y}$ を用いて
$$\boldsymbol{\delta}=\boldsymbol{y}-\tilde{\boldsymbol{X}}\tilde{\boldsymbol{\beta}}$$
と書き表せます．

* モデル変数が $$\tilde{\boldsymbol{\beta}}^\star=(\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}})^{-1}\tilde{\boldsymbol{X}}^\top\boldsymbol{y}$$ のとき，[残差平方和](https://en.wikipedia.org/wiki/Residual_sum_of_squares) $\|\boldsymbol{\delta}\|_2^2$ が最小になります．

In [ ]:
#@title （準備）描画用の関数 plot_reg を定義します（理解不要）
#import matplotlib.pyplot as plt
!pip install japanize-matplotlib -q
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

_size = 12
# 日本語フォントの設定
# グラフの文字サイズを大きめに設定 (具体的な数値を指定)
plt.rcParams['font.size'] = int(_size*1.2) # 全体のフォントサイズ
plt.rcParams['axes.labelsize'] = int(_size*1.4) # 軸ラベルのフォントサイズ
plt.rcParams['xtick.labelsize'] = int(_size) # x軸目盛りのフォントサイズ
plt.rcParams['ytick.labelsize'] = int(_size) # y軸目盛りのフォントサイズ
plt.rcParams['legend.fontsize'] = int(_size) # 凡例のフォントサイズ
plt.rcParams['figure.titlesize'] = int(_size*1.8) # 図全体のタイトルのフォントサイズ

npopts = lambda: np.printoptions(threshold=16,edgeitems=4)

def plot_reg(train, test=None, reg=None, degree=1, figsize=(5,4), s=int(_size*4), xlabel=None, ylabel=None, title=None, isGrid=True, savefig=None):
    x = train[0] if isinstance(train[0], np.ndarray) else train[0].values
    y = train[1] if isinstance(train[1], np.ndarray) else train[1].values
    xmin, xmax, dx = x.min(), x.max(), (x.max()-x.min())*0.2
    ymin, ymax, dy = y.min(), y.max(), (y.max()-y.min())*0.2

    if test is not None:
        xs = test[0] if isinstance(test[0], np.ndarray) else test[0].values
        ys = test[1] if isinstance(test[1], np.ndarray) else test[1].values
        xmin, xmax, dx = min(xmin, xs.min()), max(xmax, xs.max()), max(dx, (xs.max()-xs.min())*0.2)
        ymin, ymax, dy = min(ymin, ys[1].min()), max(ymax, ys[1].max()), max(dy, (ys[1].max()-ys[1].min())*0.2)

    #plt.figure(figsize=(3, 3.5))
    plt.figure(figsize=figsize)
    ax = plt.axes()
    ax.scatter(x, y, marker='o', c='darkviolet', s=s, zorder=10)
    if test is not None:
        ax.scatter(xs[0], ys[1], marker='x', c='0.75', s=s//2)
    if reg is not None:
        x_regr = np.linspace(xmin-dx, xmax+dx, 100)
        X_regr = PolynomialFeatures(degree).fit_transform(x_regr[:, np.newaxis])
        if isinstance(reg, (np.ndarray, list, tuple)):
            y_regr = X_regr.dot(np.asarray(reg))
        elif hasattr(reg, 'predict'):
            y_regr = reg.predict(X_regr) # モデルで予測値を計算
        else:
            y_regr = None # プロットしない
        if y_regr is not None:
            ax.plot(x_regr, y_regr, 'b-', zorder=20)

    ax.set_xlim(xmin-dx, xmax+dx)
    ax.set_ylim(ymin-dy, ymax+dy)
    plt.grid(isGrid)

    if xlabel is not None: plt.xlabel(xlabel)
    if ylabel is not None: plt.ylabel(ylabel)
    if title is not None: plt.title(title)

    plt.tight_layout()
    if savefig is not None:
        plt.savefig(savefig)


from IPython.display import display, Math
import sympy as sp
sp.init_printing()
def disp_residuals(y, X, beta, delta):
    ysp, Xsp, betasp, deltasp = sp.Matrix(y), sp.Matrix(X), sp.Matrix(beta), sp.Matrix(delta.round(1))
    yl, Xl, betal, deltal = sp.latex(ysp), sp.latex(Xsp), sp.latex(betasp), sp.latex(deltasp)
    eq = f"{deltal} = {yl} - {Xl} {betal}"
    display(Math(eq))


---
## 単回帰

モデル: $f_{\tilde{\boldsymbol{\beta}}}(x) = \beta_0 + \beta_1 x$

ひとつの説明変数 $x$ を使って，ひとつの目的変数 $y$ を予測する最もシンプルなモデルです．$x$ と $y$ の変数の間に，直線的な関係（一方が増えれば，もう一方も一定の割合で増える/減る）があると仮定します．

---
### 例：気温とアイスクリームの売上

単回帰モデルを採用する理由： 「気温が上がれば，アイスクリームの売上も伸びるだろう」という単純な仮説の検証を目的とします．売上を左右する要因は他にもありますが，まずは最も影響が大きいと思われる「気温」というひとつの要因に絞って関係性を分析します．

データ例： ある店舗における，日ごとの最高気温とアイスクリームの販売個数のデータです．

| 日 | 最高気温 (°C) (x) | アイスクリーム販売個数 (y) |
| :--- | :---: | :---: |
| 0 | 24 | 115 |
| 1 | 26 | 128 |
| 2 | 30 | 160 |
| 3 | 32 | 182 |
| 4 | 33 | 185 |
| 5 | 35 | 208 |


In [ ]:
import pandas as pd

# データの準備
data = {'最高気温 (°C) (x)': [24, 26, 30, 32, 33, 35],
        'アイスクリーム販売個数 (y)': [115, 128, 160, 182, 185, 208]}
df = pd.DataFrame(data)

# データの表を表示
print(df)

x_data = df['最高気温 (°C) (x)'].values  # 説明変数 (特徴量)
y_data = df['アイスクリーム販売個数 (y)'].values  # 目的変数 (ターゲット)

In [ ]:
plot_reg((x_data, y_data), xlabel='最高気温 (°C)', ylabel='アイスクリーム販売個数')

#### 実習課題
1. `x_data` と `y_data` をそれぞれ中心化した `xc` と `yc` を作成せよ．
2. `xc` を用いて計画行列 `X` を作成せよ．
3. `xc` から `yc` を予測する回帰直線の切片と傾きをそれぞれ $\beta_0=0$，$\beta_1=8.4$ と推定したとき，残差および残差平方和を計算せよ．
4. 残差平方和が最小になる切片と傾きを求めよ（$\tilde{\boldsymbol{\beta}}^\star=(\boldsymbol{X}^\top\boldsymbol{X})^{-1}\boldsymbol{X}^\top\boldsymbol{y}$）．
5. 中心化の有無による切片と傾きを比較せよ．

In [ ]:
# 1. x_data と y_data をそれぞれ中心化した xc と yc を作成する．
xc = x_data - x_data.mean()
yc = ''' YOUR CODE HERE '''
print(pd.DataFrame({'xc': xc, 'yc': yc}))

# 2. 計画行列 X を作成する．
X = np.column_stack((np.ones(len(xc)), xc))
print("計画行列:\n", X)

In [ ]:
# 3. xc から yc を予測する短回帰モデルの変数を beta = [0, 8.4] と推定したとき，
beta = ''' [???, ???] '''
plot_reg((xc, yc), reg=beta, xlabel='x', ylabel='y')

In [ ]:
# 残差および残差平方和を計算する．
delta = yc - X.dot(beta)
print(pd.DataFrame({'残差': delta}))

rss = ''' YOUR CODE HERE '''
print('Residual Sum of Squares: RSS =', rss)
disp_residuals(yc, X, beta, delta)

In [ ]:
# 4. 残差平方和が最小になる切片と傾きを求める
display(Math(r"$\tilde{\boldsymbol{\beta}}^\star=(\boldsymbol{X}^\top\boldsymbol{X})^{-1}\boldsymbol{X}^\top\boldsymbol{y}$"))

beta = np.linalg.inv(X.T.dot(X)).dot(X.T.dot(yc))

print(f'切片: {beta[0]:.2f}')
print(f'回帰係数 (傾き): {beta[1]:.2f}')

delta = ''' YOUR CODE HERE '''
rss = (delta**2).sum()
print('Residual Sum of Squares: RSS =', rss)

plot_reg((xc, yc), reg=beta, xlabel='x', ylabel='y')

In [ ]:
# 5. 中心化しない場合の切片と傾きを計算する．
X = ''' YOUR CODE HERE '''
beta = ''' YOUR CODE HERE '''

print(f'切片: {beta[0]:.2f}')
print(f'回帰係数 (傾き): {beta[1]:.2f}')

delta = ''' YOUR CODE HERE '''
rss = ''' YOUR CODE HERE '''
print('Residual Sum of Squares: RSS =', rss)

plot_reg((x_data, y_data), reg=beta, xlabel='x', ylabel='y')

---
## 多項式単回帰（2次の場合）

モデル: $f_{\tilde{\boldsymbol{\beta}}}(x) = \beta_0 + \beta_1 x + \beta_2 x^2$

説明変数 $x$ と目的変数 $y$ の関係が，直線ではなく曲線を描くと考えられる場合に使用します．例えば，「最初は効果が上がるが，だんだん効果が薄れ，むしろ逆効果が現れる」，「加速度的に上昇する」といった現象を表現できます．

---
### 例：肥料の量と作物の収穫量（効果が鈍化・減少する場合）

2次の多項式回帰モデルを採用する理由：畑に肥料をまくと作物の収穫量は増えますが，ある一定量を超えてまきすぎると，逆に土壌に悪影響を与え，収穫量が減少に転じることがあります．このような「山なり」の曲線的な関係を捉えるには，直線を仮定する単回帰モデルでは不十分であり，多項式モデルの方が適しています．

データ例： 10アールあたりの肥料の量と、収穫できた作物の重量のデータです．

| 試験区 | 肥料の量 (kg) (X) | 収穫量 (kg) (Y) |
| :--- | :---: | :---: |
| 0 | 10 | 450 |
| 1 | 20 | 580 |
| 2 | 30 | 650 |
| 3 | 40 | 670 |
| 4 | 50 | 640 |
| 5 | 60 | 590 |


In [ ]:
import pandas as pd

# データの準備
data = {'肥料の量 (kg) (x)': [10, 20, 30, 40, 50, 60],
             '収穫量 (kg) (y)': [450, 580, 650, 670, 640, 590]}
df = pd.DataFrame(data)

# データの表を表示
print("データ例：肥料の量と作物の収穫量")
print(df)

x_data = df['肥料の量 (kg) (x)'].values  # 説明変数 (特徴量)
y_data = df['収穫量 (kg) (y)'].values  # 目的変数 (ターゲット)

#### 実習課題
6. `x_data` を用いて計画行列 `X` を作成せよ．
7. `x_data` から `y_data` を予測する2次の多項式を $310+17x-0.2x^2$ としたとき，残差および残差平方和を計算せよ．
8. 残差平方和が最小になる2次多項式を求めよ．
9. `x_data` と `y_data` をそれぞれ標準化した `xs` と `ys` を作成せよ．
10. 標準化の有無による2次の多項式回帰を比較せよ．

In [ ]:
# 6. x_data を用いて計画行列 X を作成する．
X = np.column_stack((np.ones(len(x_data)), x_data, x_data**2))
print("計画行列:\n", X)

In [ ]:
# 7. 2次多項式 310+17*x−0.2*x**2 による予測の，
beta = ''' [???, ???, ???] '''
plot_reg((x_data, y_data), reg=beta, degree=2, xlabel='x', ylabel='y')

In [ ]:
# 残差および残差平方和を計算する．
delta = y_data - X.dot(beta)
print(pd.DataFrame({'残差': delta}))

rss = (delta**2).sum()
print('Residual Sum of Squares: RSS =', rss)

disp_residuals(y_data, X, beta, delta)

In [ ]:
# 8. 残差平方和が最小になる2次多項式を求める．
beta = np.linalg.inv(X.T.dot(X)).dot(X.T.dot(y_data))
print(pd.DataFrame({'beta': beta}))

delta = ''' YOUR CODE HERE '''
rss = (delta**2).sum()
print('Residual Sum of Squares: RSS =', rss)

plot_reg((x_data, y_data), reg=beta, degree=2, xlabel='x', ylabel='y')

In [ ]:
# 9. x_data と y_data をそれぞれ標準化した xs と ys を作成する．
xs = ''' YOUR CODE HERE '''
ys = (y_data - y_data.mean()) / y_data.std()
print(pd.DataFrame({'xs': xs, 'ys': ys}))

In [ ]:
# 10. 標準化有の2次の多項式回帰を求める．
X = np.column_stack( ( ''' YOUR CODE HERE ''') )
print("計画行列:\n", X)
beta = ''' YOUR CODE HERE '''
print(pd.DataFrame({'beta': beta}))

delta = ''' YOUR CODE HERE '''
rss = ''' YOUR CODE HERE '''
print('Residual Sum of Squares: RSS =', rss)

plot_reg((xs, ys), reg=beta, degree=2, xlabel='x', ylabel='y')